In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/quack RESEARCH/quake_project')
print(sorted(os.listdir()))

Mounted at /content/drive
['README.md', 'Untitled0.ipynb', 'build_datasets.py', 'catboost_info', 'chunks', 'comcat_global_m4.meta.json', 'comcat_global_m4.parquet', 'comcat_global_m4_features.parquet', 'comcat_global_m4_labeled.parquet', 'comcat_himalaya_m4.meta.json', 'comcat_himalaya_m4.parquet', 'comcat_himalaya_m4_features.parquet', 'comcat_himalaya_m4_labeled.parquet', 'feat_ckpt', 'figures', 'grid_ckpt', 'label_ckpt', 'labeling_summary.csv', 'models', 'scedc_socal_m25.meta.json', 'scedc_socal_m25.parquet', 'scedc_socal_m25_features.parquet', 'scedc_socal_m25_labeled.parquet', 'step10_transfer.csv', 'step10_transfer.py', 'step11_ablation.csv', 'step11_ablation_stats.py', 'step11_significance.csv', 'step12_baselines.csv', 'step12_baselines.py', 'step2_himalaya_subset.py', 'step2a_scedc_download.py', 'step3_labeling.py', 'step4_features.py', 'step5_metrics.csv', 'step5_training.py', 'step6_aci_rolling.csv', 'step6_calibration.csv', 'step6_conformal.csv', 'step6_conformal.py', 'step6

In [ ]:
!nvidia-smi                    # should show a Tesla T4
!pip install xgboost scikit-learn joblib pandas pyarrow -q

Sat Jul  4 11:09:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
CATALOGS = ["comcat_global_m4", "scedc_socal_m25", "comcat_himalaya_m4"]
RADII_KM = [25, 50, 75, 100]
WINDOWS_DAYS = [15, 30, 60, 90]
REF_LABEL = "label_R50_T30"
FEATURES = ["mw", "depth", "gap", "rms", "dmin", "magError",
            "local_count_7d", "log_energy_30d", "mag_diff_30d",
            "time_gap_local", "seismicity_z", "latitude", "longitude"]
ALPHA = 0.10
SEED = 42

MODELS_DIR = Path("./models")
GRID_CKPT = Path("./grid_ckpt")
GRID_CKPT.mkdir(exist_ok=True)

try:
    import subprocess
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    GPU = False
print(f"GPU available: {GPU}")


def all_label_cols():
    cols = [f"label_R{R}_T{T}" for R in RADII_KM for T in WINDOWS_DAYS]
    cols.append("label_adaptive")
    return cols


# ---------- reuse step-6 machinery (self-contained copies) ----------
def ece(y, p, n_bins=15):
    order = np.argsort(p)
    y, p = y[order], p[order]
    e = 0.0
    for b in np.array_split(np.arange(len(p)), n_bins):
        if len(b):
            e += (len(b) / len(p)) * abs(y[b].mean() - p[b].mean())
    return e


def split_threshold(scores_cal, alpha):
    n = len(scores_cal)
    q = np.ceil((n + 1) * (1 - alpha)) / n
    return np.quantile(scores_cal, min(q, 1.0), method="higher")


def mondrian_metrics(p_cal, y_cal, p_te, y_te, alpha=ALPHA):
    q1 = split_threshold(1.0 - p_cal[y_cal == 1], alpha)
    q0 = split_threshold(p_cal[y_cal == 0], alpha)
    in0, in1 = p_te <= q0, (1.0 - p_te) <= q1
    size = in0.astype(int) + in1.astype(int)
    covered = np.where(y_te == 1, in1, in0)
    return {
        "cov_class1": float(covered[y_te == 1].mean()) if (y_te == 1).any() else np.nan,
        "cov_class0": float(covered[y_te == 0].mean()) if (y_te == 0).any() else np.nan,
        "avg_set_size": float(size.mean()),
        "abstain_rate": float((size == 2).mean()),
    }


def isotonic_recal(p_val, y_val, p_te):
    from sklearn.isotonic import IsotonicRegression
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
    iso.fit(p_val, y_val)
    return iso.predict(p_te)


def kendall_tau(rank_a, rank_b):
    """Kendall tau between two orderings given as lists of feature names."""
    from scipy.stats import kendalltau
    common = [f for f in rank_a if f in rank_b]
    ra = [rank_a.index(f) for f in common]
    rb = [rank_b.index(f) for f in common]
    return float(kendalltau(ra, rb).statistic)


# ----------------------------------------------------------------------------
def run_config(cat, label_col, X, labels, feats, b1, b2, b3, params):
    """Train + evaluate one (catalog, label config). Returns result row
    and SHAP ranking. Checkpointed."""
    ck = GRID_CKPT / f"{cat}__{label_col}.json"
    if ck.exists():
        return json.loads(ck.read_text())

    from xgboost import XGBClassifier
    from sklearn.metrics import roc_auc_score, f1_score, recall_score, \
        accuracy_score

    y = labels[label_col].to_numpy(dtype=np.int8)
    ytr, yva, yca, yte = y[:b1], y[b1:b2], y[b2:b3], y[b3:]

    m = XGBClassifier(**params, random_state=SEED, n_jobs=-1,
                      tree_method="hist",
                      device="cuda" if GPU else "cpu",
                      eval_metric="logloss")
    m.fit(X.iloc[:b1], ytr)
    prob = m.predict_proba(X)[:, 1].astype(np.float64)
    p_va, p_ca, p_te = prob[b1:b2], prob[b2:b3], prob[b3:]

    pred = (p_te >= 0.5).astype(int)
    p_te_recal = isotonic_recal(p_va, yva, p_te)
    p_ca_recal = isotonic_recal(p_va, yva, p_ca)

    # GPU TreeSHAP global importances on a test subsample
    import xgboost as xgb
    sub = X.iloc[b3:].iloc[:20000]
    booster = m.get_booster()
    dm = xgb.DMatrix(sub)
    contribs = booster.predict(dm, pred_contribs=True)   # (n, n_feat+1)
    mean_abs = np.abs(contribs[:, :-1]).mean(axis=0)
    ranking = [feats[i] for i in np.argsort(-mean_abs)]

    row = {
        "catalog": cat, "config": label_col,
        "mainshock_rate": float(y.mean()),
        "auc": float(roc_auc_score(yte, p_te)),
        "f1": float(f1_score(yte, pred)),
        "recall": float(recall_score(yte, pred)),
        "accuracy": float(accuracy_score(yte, pred)),
        "ece_recal": float(ece(yte, p_te_recal)),
        "brier": float(np.mean((p_te - yte) ** 2)),
        **mondrian_metrics(p_ca_recal, yca, p_te_recal, yte),
        "shap_ranking": ranking,
        "shap_top1": ranking[0],
    }
    ck.write_text(json.dumps(row))
    del m, prob
    gc.collect()
    return row


def main():
    rows, shap_rows = [], []
    for cat in CATALOGS:
        print(f"\n===== {cat} =====")
        spl = np.load(MODELS_DIR / f"{cat}__splits.npz")
        b1, b2, b3 = int(spl["b1"]), int(spl["b2"]), int(spl["b3"])

        df = pd.read_parquet(f"{cat}_features.parquet")
        df = df.sort_values("time").reset_index(drop=True)
        feats = [f for f in FEATURES if f in df.columns]
        X_raw = df[feats].astype(np.float32)
        labels = df[all_label_cols()]
        ref_y = labels[REF_LABEL].to_numpy(dtype=np.int8)

        # frozen imputer + frozen hyperparameters from step 5
        import joblib
        imp = joblib.load(MODELS_DIR / f"{cat}__imputer.joblib")
        X = pd.DataFrame(imp.transform(X_raw), columns=feats)\
              .astype(np.float32)
        best = json.loads((MODELS_DIR / f"{cat}__optuna_best.json")
                          .read_text())["xgboost"]

        cfg_rows = []
        for col in all_label_cols():
            r = run_config(cat, col, X, labels, feats, b1, b2, b3, best)
            r["flip_rate_vs_ref"] = float(
                (labels[col].to_numpy(dtype=np.int8) != ref_y).mean())
            cfg_rows.append(r)
            print(f"  {col}: AUC {r['auc']:.3f} F1 {r['f1']:.3f} "
                  f"flip {r['flip_rate_vs_ref']:.3f} "
                  f"abstain {r['abstain_rate']:.3f} top1 {r['shap_top1']}")

        # SHAP stability vs reference
        ref_rank = next(r for r in cfg_rows
                        if r["config"] == REF_LABEL)["shap_ranking"]
        for r in cfg_rows:
            r["shap_kendall_tau_vs_ref"] = kendall_tau(r["shap_ranking"],
                                                       ref_rank)
            r["shap_top5_overlap"] = len(set(r["shap_ranking"][:5])
                                         & set(ref_rank[:5])) / 5.0
            shap_rows.append({"catalog": cat, "config": r["config"],
                              "ranking": " > ".join(r["shap_ranking"][:6])})
            rows.append({k: v for k, v in r.items() if k != "shap_ranking"})

        del df, X, X_raw, labels
        gc.collect()

    out = pd.DataFrame(rows).round(4)
    out.to_csv("step7_sensitivity_grid.csv", index=False)
    pd.DataFrame(shap_rows).to_csv("step7_shap_rankings.csv", index=False)

    print("\n=== step7_sensitivity_grid.csv ===")
    print(out.to_string(index=False))

    # H1 headline: spread of metrics vs spread of labels
    print("\n=== H1 summary (per catalog) ===")
    for cat in CATALOGS:
        s = out[out.catalog == cat]
        print(f"{cat}: AUC range {s.auc.min():.3f}-{s.auc.max():.3f} | "
              f"F1 range {s.f1.min():.3f}-{s.f1.max():.3f} | "
              f"flip up to {s.flip_rate_vs_ref.max():.1%} | "
              f"tau min {s.shap_kendall_tau_vs_ref.min():.2f}")


if __name__ == "__main__":
    main()

GPU available: True

===== comcat_global_m4 =====
  label_R25_T15: AUC 0.931 F1 0.894 flip 0.186 abstain 0.109 top1 mag_diff_30d
  label_R25_T30: AUC 0.928 F1 0.871 flip 0.132 abstain 0.111 top1 mag_diff_30d
  label_R25_T60: AUC 0.916 F1 0.834 flip 0.139 abstain 0.185 top1 mag_diff_30d
  label_R25_T90: AUC 0.912 F1 0.806 flip 0.149 abstain 0.189 top1 mag_diff_30d
  label_R50_T15: AUC 0.948 F1 0.906 flip 0.078 abstain 0.076 top1 mag_diff_30d
  label_R50_T30: AUC 0.955 F1 0.894 flip 0.000 abstain 0.042 top1 mag_diff_30d
  label_R50_T60: AUC 0.950 F1 0.822 flip 0.087 abstain 0.053 top1 mag_diff_30d
  label_R50_T90: AUC 0.948 F1 0.778 flip 0.139 abstain 0.056 top1 mag_diff_30d
  label_R75_T15: AUC 0.930 F1 0.842 flip 0.107 abstain 0.146 top1 mag_diff_30d
  label_R75_T30: AUC 0.939 F1 0.803 flip 0.085 abstain 0.095 top1 mag_diff_30d


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [11:11:08] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  label_R75_T60: AUC 0.941 F1 0.728 flip 0.175 abstain 0.100 top1 mag_diff_30d
  label_R75_T90: AUC 0.942 F1 0.679 flip 0.224 abstain 0.085 top1 mag_diff_30d
  label_R100_T15: AUC 0.922 F1 0.792 flip 0.137 abstain 0.152 top1 mag_diff_30d
  label_R100_T30: AUC 0.935 F1 0.744 flip 0.145 abstain 0.110 top1 mag_diff_30d
  label_R100_T60: AUC 0.941 F1 0.675 flip 0.232 abstain 0.100 top1 mag_diff_30d
  label_R100_T90: AUC 0.944 F1 0.627 flip 0.278 abstain 0.074 top1 mag_diff_30d
  label_adaptive: AUC 0.922 F1 0.755 flip 0.158 abstain 0.153 top1 mag_diff_30d

===== scedc_socal_m25 =====
  label_R25_T15: AUC 0.954 F1 0.879 flip 0.129 abstain 0.023 top1 mag_diff_30d
  label_R25_T30: AUC 0.957 F1 0.872 flip 0.078 abstain 0.025 top1 mag_diff_30d
  label_R25_T60: AUC 0.957 F1 0.857 flip 0.075 abstain 0.038 top1 mag_diff_30d
  label_R25_T90: AUC 0.960 F1 0.853 flip 0.077 abstain 0.020 top1 mag_diff_30d
  label_R50_T15: AUC 0.969 F1 0.897 flip 0.052 abstain 0.000 top1 mag_diff_30d
  label_R50_T30: A

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

CAT = "comcat_global_m4"
MODEL = "xgboost"
REF_LABEL = "label_R50_T30"
ALPHA_GRID = np.round(np.arange(0.005, 0.31, 0.005), 3)
ALPHA_STAR_REPORT = 0.02
COST_FN = [5, 10, 50]
MODELS_DIR = Path("./models")

CASES = {
    "Ridgecrest_2019": dict(t0="2019-06-28", t1="2019-08-10",
                            lat=(35.0, 36.5), lon=(-118.2, -117.0)),
    "Kumamoto_2016":   dict(t0="2016-04-10", t1="2016-05-15",
                            lat=(32.0, 33.6), lon=(130.0, 131.5)),
    "Kahramanmaras_2023": dict(t0="2023-01-30", t1="2023-03-15",
                               lat=(36.0, 39.0), lon=(35.5, 39.0)),
}


# ----------------------------------------------------------------------------
def split_threshold(scores_cal, alpha):
    n = len(scores_cal)
    if n == 0:
        return 1.0
    q = np.ceil((n + 1) * (1 - alpha)) / n
    return np.quantile(scores_cal, min(q, 1.0), method="higher")


def mondrian_sets_nonempty(p, q0, q1):
    in0 = p <= q0
    in1 = (1.0 - p) <= q1
    empty = ~(in0 | in1)
    # force argmax on empty sets
    in1 = in1 | (empty & (p >= 0.5))
    in0 = in0 | (empty & (p < 0.5))
    return in0, in1, empty


def actions(in0, in1):
    both = in0 & in1
    only1 = in1 & ~in0
    only0 = in0 & ~in1
    a = np.where(both, "ESCALATE", np.where(only1, "RESPONSE", "ALERT"))
    return a, only1, only0, both


# ----------------------------------------------------------------------------
def main():
    spl = np.load(MODELS_DIR / f"{CAT}__splits.npz")
    b1, b2, b3 = int(spl["b1"]), int(spl["b2"]), int(spl["b3"])

    df = pd.read_parquet(f"{CAT}_features.parquet",
                         columns=["time", "mw", "place", "latitude",
                                  "longitude", REF_LABEL])
    df = df.sort_values("time").reset_index(drop=True)
    y = df[REF_LABEL].to_numpy(np.int8)
    p = np.load(MODELS_DIR / f"{CAT}__{MODEL}__recal_probs.npz")["prob"]\
          .astype(np.float64)

    p_cal, y_cal = p[b2:b3], y[b2:b3]
    p_te, y_te = p[b3:], y[b3:]

    # ---------------- C) cost analysis over alpha ----------------
    rows = []
    for alpha in ALPHA_GRID:
        q1 = split_threshold(1 - p_cal[y_cal == 1], alpha)
        q0 = split_threshold(p_cal[y_cal == 0], alpha)
        in0, in1, empty = mondrian_sets_nonempty(p_te, q0, q1)
        _, only1, only0, both = actions(in0, in1)
        fn = float((only1 & (y_te == 0)).mean())   # declared safe, wasn't
        fp = float((only0 & (y_te == 1)).mean())   # alert on true mainshock
        ab = float(both.mean())
        for c in COST_FN:
            rows.append({"alpha": alpha, "c_FN": c,
                         "P_false_response": round(fn, 4),
                         "P_false_alert": round(fp, 4),
                         "P_escalate": round(ab, 4),
                         "P_empty_raw": round(float(empty.mean()), 4),
                         "expected_cost": round(c * fn + fp + ab, 4)})
    costs = pd.DataFrame(rows)
    costs.to_csv("step8_decision_costs.csv", index=False)
    print("=== alpha* per cost ratio (test block) ===")
    for c in COST_FN:
        sub = costs[costs.c_FN == c]
        best = sub.loc[sub.expected_cost.idxmin()]
        print(f"  c_FN={c:>2}: alpha*={best.alpha:.2f} "
              f"(cost {best.expected_cost:.4f}, "
              f"escalate {best.P_escalate:.1%}, "
              f"false-response {best.P_false_response:.2%})")

    # ---------------- D) case-study replays ----------------
    case_rows = []
    times = df["time"]
    for name, c in CASES.items():
        mask = (times.between(c["t0"], c["t1"])
                & df["latitude"].between(*c["lat"])
                & df["longitude"].between(*c["lon"])
                & (df["mw"] >= 4.5))
        idx = np.where(mask.to_numpy())[0]
        if len(idx) == 0:
            print(f"\n[{name}] no events found — check bbox")
            continue
        block = ("train" if idx[0] < b1 else
                 "val" if idx[0] < b2 else
                 "calib" if idx[0] < b3 else "test")
        # exclude the sequence itself from calibration if it lies in calib
        cal_mask = np.ones(b3 - b2, dtype=bool)
        in_cal = idx[(idx >= b2) & (idx < b3)] - b2
        cal_mask[in_cal] = False
        pc, yc = p_cal[cal_mask], y_cal[cal_mask]
        q1 = split_threshold(1 - pc[yc == 1], ALPHA_STAR_REPORT)
        q0 = split_threshold(pc[yc == 0], ALPHA_STAR_REPORT)
        in0, in1, _ = mondrian_sets_nonempty(p[idx], q0, q1)
        act, _, _, _ = actions(in0, in1)
        print(f"\n=== {name} (falls in {block.upper()} block"
              f"{'' if block in ('calib', 'test') else ' — IN-SAMPLE, flag in paper'}) ===")
        sub = df.iloc[idx][["time", "mw", "place"]].copy()
        sub["prob"] = np.round(p[idx], 3)
        sub["set"] = ["{0,1}" if a and b else ("{1}" if b else "{0}")
                      for a, b in zip(in0, in1)]
        sub["action"] = act
        sub["true_label"] = y[idx]
        sub["sequence"] = name
        sub["block"] = block
        show = sub[sub.mw >= 5.0] if len(sub) > 25 else sub
        print(show.drop(columns=["sequence", "block"])
              .to_string(index=False))
        case_rows.append(sub)

    if case_rows:
        pd.concat(case_rows).to_csv("step8_case_studies.csv", index=False)
        print("\nsaved -> step8_case_studies.csv (full event lists)")


if __name__ == "__main__":
    main()

=== alpha* per cost ratio (test block) ===
  c_FN= 5: alpha*=0.09 (cost 0.3326, escalate 7.7%, false-response 4.32%)
  c_FN=10: alpha*=0.03 (cost 0.4541, escalate 31.7%, false-response 1.26%)
  c_FN=50: alpha*=0.01 (cost 0.5835, escalate 44.7%, false-response 0.27%)

=== Ridgecrest_2019 (falls in CALIB block) ===
                            time   mw                          place  prob   set   action  true_label
       2019-07-04 17:33:49+00:00 6.40 Ridgecrest Earthquake Sequence 0.976   {1} RESPONSE           0
2019-07-04 18:39:44.440000+00:00 4.59      7km ESE of Ridgecrest, CA 0.001   {0}    ALERT           0
2019-07-04 18:56:06.420000+00:00 4.58      15km NE of Ridgecrest, CA 0.001   {0}    ALERT           0
2019-07-04 19:21:32.090000+00:00 4.50 13km SSW of Searles Valley, CA 0.001   {0}    ALERT           0
2019-07-05 11:07:53.040000+00:00 5.37   16km W of Searles Valley, CA 0.001   {0}    ALERT           0
2019-07-06 03:16:32.480000+00:00 4.97 14km WSW of Searles Valley, CA 0.00

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
    "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
})

FIG = Path("./figures")
FIG.mkdir(exist_ok=True)
CATS = ["comcat_global_m4", "scedc_socal_m25", "comcat_himalaya_m4"]
NICE = {"comcat_global_m4": "Global (ComCat, M\u22654)",
        "scedc_socal_m25": "S. California (SCEDC, M\u22652.5)",
        "comcat_himalaya_m4": "Himalaya (ComCat, M\u22654)"}
RADII, WINDOWS = [25, 50, 75, 100], [15, 30, 60, 90]
MODELS_DIR = Path("./models")


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"saved {name}")


def grid_from(df, value):
    g = np.full((len(RADII), len(WINDOWS)), np.nan)
    for i, R in enumerate(RADII):
        for j, T in enumerate(WINDOWS):
            row = df[df.config == f"label_R{R}_T{T}"]
            if len(row):
                g[i, j] = row.iloc[0][value]
    return g


def heatmap(ax, g, title, fmt="{:.2f}", cmap="viridis"):
    im = ax.imshow(g, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(WINDOWS)), [f"{t}d" for t in WINDOWS])
    ax.set_yticks(range(len(RADII)), [f"{r}km" for r in RADII])
    ax.set_xlabel("time window T")
    ax.set_ylabel("radius R")
    ax.set_title(title)
    for i in range(g.shape[0]):
        for j in range(g.shape[1]):
            if not np.isnan(g[i, j]):
                ax.text(j, i, fmt.format(g[i, j]), ha="center",
                        va="center", fontsize=7,
                        color="white" if g[i, j] < np.nanmean(g) else "black")
    return im


# ---------------- fig 1: completeness + splits ----------------
def fig1():
    fig, axes = plt.subplots(1, 3, figsize=(11, 3))
    for ax, cat in zip(axes, CATS):
        t = pd.read_parquet(f"{cat}_features.parquet", columns=["time"])
        t = t.sort_values("time").reset_index(drop=True)
        yr = t["time"].dt.year
        counts = yr.value_counts().sort_index()
        ax.fill_between(counts.index, counts.values, color="#4C72B0",
                        alpha=0.8)
        spl = np.load(MODELS_DIR / f"{cat}__splits.npz")
        for b, lab in [(int(spl["b1"]), "train|val"),
                       (int(spl["b2"]), "val|calib"),
                       (int(spl["b3"]), "calib|test")]:
            xb = yr.iloc[b]
            ax.axvline(xb, color="crimson", ls="--", lw=1)
            ax.text(xb, ax.get_ylim()[1] * 0.95, str(xb), rotation=90,
                    fontsize=6, color="crimson", va="top")
        ax.set_title(NICE[cat])
        ax.set_xlabel("year")
        ax.set_ylabel("events / year")
    save(fig, "fig1_completeness_splits")


# ---------------- fig 2 & 3: label-rate and flip heatmaps ----------------
def fig2_fig3():
    lab = pd.read_csv("labeling_summary.csv")
    for name, value, cmap, fname in [
            ("mainshock rate", "mainshock_rate", "viridis",
             "fig2_label_rate_heatmaps"),
            ("label flip rate vs (50 km, 30 d)", "flip_rate_vs_ref",
             "magma", "fig3_flip_rate_heatmaps")]:
        fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
        for ax, cat in zip(axes, CATS):
            g = grid_from(lab[lab.catalog == cat], value)
            heatmap(ax, g, NICE[cat], cmap=cmap)
        fig.suptitle(name, y=1.03)
        save(fig, fname)


# ---------------- fig 4: reliability diagrams ----------------
def fig4():
    rel = pd.read_csv("step6_reliability_data.csv")
    cal = pd.read_csv("step6_calibration.csv")
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
    for ax, cat in zip(axes, CATS):
        sub = cal[(cal.catalog == cat) & (cal.model == "xgboost")]
        best = sub[sub.variant != "raw"].nsmallest(1, "ece").iloc[0]
        for variant, color, ls in [("raw", "#C44E52", "-"),
                                   (best.variant, "#4C72B0", "-")]:
            r = rel[(rel.catalog == cat) & (rel.model == "xgboost")
                    & (rel.variant == variant)]
            e = sub[sub.variant == variant].iloc[0].ece
            ax.plot(r.mean_pred, r.frac_pos, marker="o", ms=3, ls=ls,
                    color=color,
                    label=f"{variant} (ECE {e:.3f})")
        ax.plot([0, 1], [0, 1], color="gray", lw=0.8, ls=":")
        ax.set_title(NICE[cat])
        ax.set_xlabel("mean predicted probability")
        ax.set_ylabel("observed mainshock fraction")
        ax.legend(fontsize=7, loc="upper left")
    fig.suptitle("Reliability (XGBoost, test block)", y=1.03)
    save(fig, "fig4_reliability")


# ---------------- fig 5: rolling coverage split vs ACI ----------------
def fig5():
    aci = pd.read_csv("step6_aci_rolling.csv")
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
    for ax, cat in zip(axes, CATS):
        r = aci[(aci.catalog == cat) & (aci.model == "xgboost")]
        ax.plot(r.t, r.rolling_cov_split, color="#C44E52",
                label="split CP (fixed)")
        ax.plot(r.t, r.rolling_cov_aci, color="#4C72B0", label="ACI")
        ax.axhline(0.90, color="gray", ls=":", lw=1, label="target 0.90")
        ax.set_title(NICE[cat])
        ax.set_xlabel("test-stream index (chronological)")
        ax.set_ylim(0.80, 1.0)
    axes[0].set_ylabel("rolling coverage (w=2000)")
    axes[0].legend(fontsize=7)
    fig.suptitle("Coverage under temporal drift (\u03b1 = 0.10)", y=1.03)
    save(fig, "fig5_aci_rolling")


# ---------------- fig 6: H1 — AUC stable, F1 not ----------------
def fig6():
    g = pd.read_csv("step7_sensitivity_grid.csv")
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
    for ax, cat in zip(axes, CATS):
        s = g[g.catalog == cat]
        fixed = s[s.config != "label_adaptive"]
        adapt = s[s.config == "label_adaptive"]
        sc = ax.scatter(fixed.flip_rate_vs_ref, fixed.f1, c=fixed.auc,
                        cmap="viridis", s=45, vmin=0.87, vmax=0.99,
                        label="fixed windows")
        ax.scatter(adapt.flip_rate_vs_ref, adapt.f1, marker="*", s=180,
                   c=adapt.auc, cmap="viridis", vmin=0.87, vmax=0.99,
                   edgecolor="crimson", linewidth=1.2,
                   label="adaptive window")
        ax.set_title(NICE[cat])
        ax.set_xlabel("label flip rate vs reference")
        ax.set_ylabel("F1 (test)")
        ax.legend(fontsize=7, loc="lower left")
    cb = fig.colorbar(sc, ax=axes, shrink=0.85)
    cb.set_label("AUC")
    fig.suptitle("H1: discrimination (AUC) is window-stable, "
                 "decisions (F1) are not", y=1.03)
    fig.savefig(FIG / "fig6_h1_sensitivity.png", bbox_inches="tight")
    fig.savefig(FIG / "fig6_h1_sensitivity.pdf", bbox_inches="tight")
    plt.close(fig)
    print("saved fig6_h1_sensitivity")


# ---------------- fig 7: abstention gradient across catalogs ----------------
def fig7():
    c = pd.read_csv("step6_conformal.csv")
    c = c[(c.method == "mondrian") & (c.scores == "recal")
          & (c.model == "xgboost")]
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))
    x = np.arange(len(CATS))
    width = 0.25
    for k, alpha in enumerate([0.05, 0.10, 0.20]):
        sub = c[c.alpha == alpha].set_index("catalog").loc[CATS]
        axes[0].bar(x + (k - 1) * width, sub.abstain_rate, width,
                    label=f"\u03b1={alpha}")
        axes[1].bar(x + (k - 1) * width, sub.avg_set_size, width,
                    label=f"\u03b1={alpha}")
    for ax, ylab in zip(axes, ["abstention rate", "avg prediction-set size"]):
        ax.set_xticks(x, [NICE[c].split(" (")[0] for c in CATS],
                      fontsize=8)
        ax.set_ylabel(ylab)
        ax.legend(fontsize=7)
    fig.suptitle("Honest uncertainty scales with catalog completeness "
                 "(Mondrian, XGBoost)", y=1.03)
    save(fig, "fig7_trust_gradient")


# ---------------- fig 8: Kahramanmaras replay timeline ----------------
def fig8():
    cs = pd.read_csv("step8_case_studies.csv", parse_dates=["time"])
    cs = cs[cs.sequence == "Kahramanmaras_2023"].sort_values("time")
    colors = {"RESPONSE": "#2E8B57", "ALERT": "#E6A700",
              "ESCALATE": "#C44E52"}
    fig, ax = plt.subplots(figsize=(9, 3.6))
    for act, col in colors.items():
        s = cs[cs.action == act]
        ax.scatter(s.time, s.mw, s=(s.mw - 4.0) * 60, color=col,
                   label=act, zorder=3,
                   edgecolor="black", linewidth=0.4)
    wrong = cs[(cs.action == "RESPONSE") & (cs.true_label == 0)]
    ax.scatter(wrong.time, wrong.mw, marker="x", color="black", s=70,
               zorder=4, label="incorrect RESPONSE")
    for _, r in cs[cs.mw >= 6.3].iterrows():
        ax.annotate(f"M{r.mw:.1f}", (r.time, r.mw),
                    textcoords="offset points", xytext=(0, 7),
                    ha="center", fontsize=7)
    ax.set_ylabel("Mw")
    ax.set_xlabel("2023")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.legend(fontsize=7, ncol=4, loc="upper right")
    ax.set_title("Kahramanmara\u015f 2023 replay (out-of-sample, "
                 "Mondrian \u03b1 = 0.10)")
    save(fig, "fig8_case_study_timeline")


if __name__ == "__main__":
    for f in [fig1, fig2_fig3, fig4, fig5, fig6, fig7, fig8]:
        try:
            f()
        except Exception as e:
            print(f"[warn] {f.__name__} failed: {type(e).__name__}: {e}")
    print("\nAll figures in ./figures/")

saved fig1_completeness_splits
saved fig2_label_rate_heatmaps
saved fig3_flip_rate_heatmaps
saved fig4_reliability
saved fig5_aci_rolling
saved fig6_h1_sensitivity
saved fig7_trust_gradient
saved fig8_case_study_timeline

All figures in ./figures/


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9.5,
    "axes.titlesize": 10.5, "axes.labelsize": 9.5,
    "axes.spines.top": False, "axes.spines.right": False,
})

FIG = Path("./figures")
FIG.mkdir(exist_ok=True)
CATS = ["comcat_global_m4", "scedc_socal_m25", "comcat_himalaya_m4"]
NICE = {"comcat_global_m4": "Global (ComCat, M\u22654.0)",
        "scedc_socal_m25": "S. California (SCEDC, M\u22652.5)",
        "comcat_himalaya_m4": "Himalayan arc (ComCat, M\u22654.0)"}
SHORT = {"comcat_global_m4": "Global", "scedc_socal_m25": "S. California",
         "comcat_himalaya_m4": "Himalaya"}
RADII, WINDOWS = [25, 50, 75, 100], [15, 30, 60, 90]
MODELS_DIR = Path("./models")


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"saved {name}")


def date_axis(ax):
    """Robust, readable date ticks."""
    loc = mdates.AutoDateLocator(minticks=4, maxticks=8)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))


def grid_from(df, value):
    g = np.full((4, 4), np.nan)
    for i, R in enumerate(RADII):
        for j, T in enumerate(WINDOWS):
            r = df[df.config == f"label_R{R}_T{T}"]
            if len(r):
                g[i, j] = r.iloc[0][value]
    return g


def heatmap(ax, g, title, cmap):
    ax.imshow(g, cmap=cmap, aspect="auto")
    ax.set_xticks(range(4), [f"{t} d" for t in WINDOWS])
    ax.set_yticks(range(4), [f"{r} km" for r in RADII])
    ax.set_xlabel("temporal half-window $T$")
    ax.set_title(title)
    vmid = (np.nanmax(g) + np.nanmin(g)) / 2
    for i in range(4):
        for j in range(4):
            if not np.isnan(g[i, j]):
                ax.text(j, i, f"{g[i, j]:.2f}", ha="center", va="center",
                        fontsize=8,
                        color="white" if g[i, j] < vmid else "black")
    ax.add_patch(plt.Rectangle((0.5, 0.5), 1, 1, fill=False,
                               edgecolor="red", lw=2))   # (R50, T30)


# ---------------- fig 1 ----------------
def fig1():
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.2))
    for ax, cat in zip(axes, CATS):
        t = pd.read_parquet(f"{cat}_features.parquet", columns=["time"])
        t["time"] = pd.to_datetime(t["time"], utc=True)
        t = t.sort_values("time").reset_index(drop=True)
        yr = t["time"].dt.year
        counts = yr.value_counts().sort_index()
        ax.fill_between(counts.index, counts.values, color="#4C72B0",
                        alpha=0.85)
        spl = np.load(MODELS_DIR / f"{cat}__splits.npz")
        for b in [int(spl["b1"]), int(spl["b2"]), int(spl["b3"])]:
            ax.axvline(yr.iloc[b], color="crimson", ls="--", lw=1)
        ax.set_title(NICE[cat])
        ax.set_xlabel("year")
    axes[0].set_ylabel("events / year")
    fig.suptitle("Catalog completeness over time; dashed lines: "
                 "train | val | calib | test boundaries", y=1.04)
    fig.tight_layout()
    save(fig, "fig1_completeness_splits")


# ---------------- figs 2 & 3 ----------------
def fig2_fig3():
    lab = pd.read_csv("labeling_summary.csv")
    specs = [("Mainshock rate as a function of the labeling window "
              "(red box: reference)", "mainshock_rate", "viridis",
              "fig2_label_rate_heatmaps"),
             ("Fraction of labels that flip relative to the reference "
              "window", "flip_rate_vs_ref", "magma",
              "fig3_flip_rate_heatmaps")]
    for title, value, cmap, fname in specs:
        fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4))
        for ax, cat in zip(axes, CATS):
            heatmap(ax, grid_from(lab[lab.catalog == cat], value),
                    NICE[cat], cmap)
        axes[0].set_ylabel("spatial radius $R$")
        fig.suptitle(title, y=1.04)
        fig.tight_layout()
        save(fig, fname)


# ---------------- fig 4 ----------------
def fig4():
    rel = pd.read_csv("step6_reliability_data.csv")
    cal = pd.read_csv("step6_calibration.csv")
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.5))
    for ax, cat in zip(axes, CATS):
        sub = cal[(cal.catalog == cat) & (cal.model == "xgboost")]
        best = sub[sub.variant != "raw"].nsmallest(1, "ece").iloc[0]
        for variant, color in [("raw", "#C44E52"),
                               (best.variant, "#4C72B0")]:
            r = rel[(rel.catalog == cat) & (rel.model == "xgboost")
                    & (rel.variant == variant)]
            e = sub[sub.variant == variant].iloc[0].ece
            ax.plot(r.mean_pred, r.frac_pos, marker="o", ms=3.5,
                    color=color, label=f"{variant} (ECE {e:.3f})")
        ax.plot([0, 1], [0, 1], color="gray", lw=0.8, ls=":")
        ax.set_title(NICE[cat])
        ax.set_xlabel("mean predicted probability")
        ax.legend(fontsize=7.5, loc="upper left")
    axes[0].set_ylabel("observed mainshock fraction")
    fig.suptitle("Reliability diagrams (XGBoost, test block)", y=1.04)
    fig.tight_layout()
    save(fig, "fig4_reliability")


# ---------------- fig 5 ----------------
def fig5():
    aci = pd.read_csv("step6_aci_rolling.csv")
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.3), sharey=True)
    for ax, cat in zip(axes, CATS):
        r = aci[(aci.catalog == cat) & (aci.model == "xgboost")]
        ax.plot(r.t, r.rolling_cov_split, color="#C44E52", lw=1.2,
                label="split CP (fixed threshold)")
        ax.plot(r.t, r.rolling_cov_aci, color="#4C72B0", lw=1.2,
                label="ACI (adaptive)")
        ax.axhline(0.90, color="gray", ls=":", lw=1)
        ax.set_title(NICE[cat])
        ax.set_xlabel("chronological test-stream index")
    axes[0].set_ylabel("rolling coverage (window = 2000)")
    axes[0].legend(fontsize=7.5, loc="lower left")
    fig.suptitle("Empirical coverage under temporal drift, target 0.90 "
                 "(\u03b1 = 0.10)", y=1.04)
    fig.tight_layout()
    save(fig, "fig5_aci_rolling")


# ---------------- fig 6 ----------------
def fig6():
    g = pd.read_csv("step7_sensitivity_grid.csv")
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), sharey=True)
    for ax, cat in zip(axes, CATS):
        s = g[g.catalog == cat]
        fx = s[s.config != "label_adaptive"]
        ad = s[s.config == "label_adaptive"]
        rf = s[s.config == "label_R50_T30"]
        sc = ax.scatter(fx.flip_rate_vs_ref * 100, fx.f1, c=fx.auc,
                        cmap="viridis", s=55, vmin=0.88, vmax=0.98,
                        edgecolor="k", linewidth=0.3,
                        label="fixed windows")
        ax.scatter(ad.flip_rate_vs_ref * 100, ad.f1, marker="*", s=260,
                   c=ad.auc, cmap="viridis", vmin=0.88, vmax=0.98,
                   edgecolor="crimson", linewidth=1.4, zorder=5,
                   label="adaptive window")
        ax.scatter(rf.flip_rate_vs_ref * 100, rf.f1, facecolors="none",
                   edgecolor="red", s=180, linewidth=1.6, zorder=4,
                   label="reference")
        ax.set_title(NICE[cat])
        ax.set_xlabel("label flip rate vs reference (%)")
    axes[0].set_ylabel("F1 score (test block)")
    axes[0].legend(fontsize=7.5, loc="lower left")
    cb = fig.colorbar(sc, ax=axes, shrink=0.85, pad=0.015)
    cb.set_label("AUC")
    fig.suptitle("Ranking quality (AUC, color) is window-stable; "
                 "thresholded decisions (F1) are not", y=1.04)
    save(fig, "fig6_h1_sensitivity")


# ---------------- fig 7 ----------------
def fig7():
    conf = pd.read_csv("step6_conformal.csv")
    conf = conf[(conf.scores == "recal") & (conf.alpha == 0.10)]
    m = conf[conf.method == "mondrian"]
    s = conf[conf.method == "split"]
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4))
    x = np.arange(3)
    w = 0.24
    for k, (mod, col) in enumerate(zip(
            ["catboost", "xgboost", "lightgbm"],
            ["#4C72B0", "#DD8452", "#55A868"])):
        sub = m[m.model == mod].set_index("catalog").reindex(CATS)
        axes[0].bar(x + (k - 1) * w, sub.abstain_rate * 100, w,
                    color=col, label=mod)
        axes[1].bar(x + (k - 1) * w, sub.avg_set_size, w, color=col)
    ss = s[s.model == "xgboost"].set_index("catalog").reindex(CATS)
    sm = m[m.model == "xgboost"].set_index("catalog").reindex(CATS)
    axes[2].bar(x - 0.15, ss.coverage_class0 * 100, 0.3,
                color="#C44E52", label="split CP")
    axes[2].bar(x + 0.15, sm.coverage_class0 * 100, 0.3,
                color="#4C72B0", label="Mondrian")
    axes[2].axhline(90, color="gray", ls=":", lw=1)
    axes[2].set_ylim(75, 101)
    for ax, t in zip(axes, ["Abstention rate (%)",
                            "Mean prediction-set size",
                            "Class-0 coverage (%), XGBoost"]):
        ax.set_xticks(x, [SHORT[c] for c in CATS])
        ax.set_title(t, fontsize=9.5)
    axes[1].axhline(1.0, color="gray", ls=":", lw=1)
    axes[0].legend(fontsize=7.5)
    axes[2].legend(fontsize=7.5, loc="lower left")
    fig.suptitle("Mondrian conformal at \u03b1 = 0.10: uncertainty grows "
                 "as completeness falls; per-class coverage is restored",
                 y=1.05)
    fig.tight_layout()
    save(fig, "fig7_trust_gradient")


# ---------------- fig 8 (FIXED) ----------------
def fig8():
    cs = pd.read_csv("step8_case_studies.csv")
    # ROBUST datetime parsing — this was the categorical-axis bug
    cs["time"] = pd.to_datetime(cs["time"], utc=True, errors="coerce",
                                format="mixed")
    cs = cs.dropna(subset=["time"])
    cs = cs[cs.sequence == "Kahramanmaras_2023"].sort_values("time")

    colors = {"RESPONSE": "#2E8B57", "ALERT": "#E6A700",
              "ESCALATE": "#C44E52"}
    fig, ax = plt.subplots(figsize=(10, 4))
    for act, col in colors.items():
        s = cs[cs.action == act]
        ax.scatter(s.time.to_numpy(), s.mw, s=(s.mw - 4.2) * 70,
                   color=col, label=act, zorder=3,
                   edgecolor="black", linewidth=0.4)
    wrong = cs[((cs.action == "RESPONSE") & (cs.true_label == 0)) |
               ((cs.action == "ALERT") & (cs.true_label == 1))]
    ax.scatter(wrong.time.to_numpy(), wrong.mw, marker="x",
               color="black", s=90, zorder=4, label="decision \u2260 label")
    for _, r in cs[cs.mw >= 6.3].iterrows():
        ax.annotate(f"M{r.mw:.1f}", (r.time, r.mw),
                    textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=8)
    date_axis(ax)
    ax.set_ylabel("moment magnitude $M_w$")
    ax.set_ylim(4.4, 8.3)
    ax.legend(fontsize=8, ncol=4, loc="upper right")
    ax.set_title("Kahramanmara\u015f 2023 replay (out-of-sample; "
                 "Mondrian sets, \u03b1 = 0.10)")
    fig.tight_layout()
    save(fig, "fig8_case_study_timeline")


# ---------------- fig 9 ----------------
def fig9():
    c = pd.read_csv("step8_decision_costs.csv")
    fig, ax = plt.subplots(figsize=(6.2, 3.6))
    for cfn, col in zip(sorted(c.c_FN.unique()),
                        ["#4C72B0", "#DD8452", "#C44E52"]):
        s = c[c.c_FN == cfn].sort_values("alpha")
        ax.plot(s.alpha, s.expected_cost, marker="o", ms=3.5, color=col,
                label=f"$c_{{FN}}/c_{{FP}}$ = {cfn}")
        best = s.loc[s.expected_cost.idxmin()]
        ax.scatter(best.alpha, best.expected_cost, s=110,
                   facecolors="none", edgecolor=col, linewidth=1.6,
                   zorder=5)
    ax.set_xlabel("conformal risk level \u03b1")
    ax.set_ylabel("expected operational cost")
    ax.legend(fontsize=8)
    ax.set_title("Cost-based selection of the operating risk level "
                 "(circles: \u03b1*)")
    fig.tight_layout()
    save(fig, "fig9_decision_costs")


# ---------------- fig 10 ----------------
def fig10():
    g = pd.read_csv("step7_sensitivity_grid.csv")
    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    markers = {"comcat_global_m4": "o", "scedc_socal_m25": "s",
               "comcat_himalaya_m4": "^"}
    for cat in CATS:
        s = g[(g.catalog == cat) & (g.config != "label_R50_T30")]
        ax.scatter(s.flip_rate_vs_ref * 100, s.shap_kendall_tau_vs_ref,
                   marker=markers[cat], s=45, alpha=0.85,
                   label=SHORT[cat])
    ax.axhline(1.0, color="gray", ls=":", lw=1)
    ax.set_xlabel("label flip rate vs reference (%)")
    ax.set_ylabel("Kendall \u03c4 of SHAP ranking vs reference")
    ax.legend(fontsize=8)
    ax.set_title("Interpretation stability: SHAP rankings reshuffle "
                 "as the labeling window moves")
    fig.tight_layout()
    save(fig, "fig10_shap_stability")


if __name__ == "__main__":
    for f in [fig1, fig2_fig3, fig4, fig5, fig6, fig7, fig8, fig9, fig10]:
        try:
            f()
        except Exception as e:
            print(f"[warn] {f.__name__} failed: {type(e).__name__}: {e}")
    print("\nAll figures in ./figures/ (.png + .pdf)")

saved fig1_completeness_splits
saved fig2_label_rate_heatmaps
saved fig3_flip_rate_heatmaps
saved fig4_reliability
saved fig5_aci_rolling
saved fig6_h1_sensitivity
saved fig7_trust_gradient
saved fig8_case_study_timeline
saved fig9_decision_costs
saved fig10_shap_stability

All figures in ./figures/ (.png + .pdf)


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

REF_LABEL = "label_R50_T30"
FEATURES = ["mw", "depth", "gap", "rms", "dmin", "magError",
            "local_count_7d", "log_energy_30d", "mag_diff_30d",
            "time_gap_local", "seismicity_z", "latitude", "longitude"]
ALPHA = 0.10
SEED = 42
BOX = dict(min_lat=20.0, max_lat=32.0, min_lon=85.0, max_lon=98.0)
MODELS_DIR = Path("./models")

try:
    import subprocess
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    GPU = False
print(f"GPU available: {GPU}")


# ---------------- shared machinery ----------------
def ece(y, p, n_bins=15):
    order = np.argsort(p)
    y, p = y[order], p[order]
    e = 0.0
    for b in np.array_split(np.arange(len(p)), n_bins):
        if len(b):
            e += (len(b) / len(p)) * abs(y[b].mean() - p[b].mean())
    return float(e)


def isotonic_fit(p, y):
    from sklearn.isotonic import IsotonicRegression
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
    iso.fit(p, y)
    return iso


def split_threshold(scores, alpha):
    n = len(scores)
    if n == 0:
        return 1.0
    q = np.ceil((n + 1) * (1 - alpha)) / n
    return np.quantile(scores, min(q, 1.0), method="higher")


def mondrian_eval(p_cal, y_cal, p_te, y_te, alpha=ALPHA):
    q1 = split_threshold(1 - p_cal[y_cal == 1], alpha)
    q0 = split_threshold(p_cal[y_cal == 0], alpha)
    in0, in1 = p_te <= q0, (1 - p_te) <= q1
    size = in0.astype(int) + in1.astype(int)
    cov = np.where(y_te == 1, in1, in0)
    return {
        "cov_class1": float(cov[y_te == 1].mean()) if (y_te == 1).any() else np.nan,
        "cov_class0": float(cov[y_te == 0].mean()) if (y_te == 0).any() else np.nan,
        "avg_set_size": float(size.mean()),
        "abstain_rate": float((size == 2).mean()),
    }


def discrimination(y, p):
    from sklearn.metrics import roc_auc_score, f1_score
    return {"auc": float(roc_auc_score(y, p)),
            "f1": float(f1_score(y, (p >= 0.5).astype(int))),
            "ece": ece(y, p),
            "brier": float(np.mean((p - y) ** 2))}


def in_box(df):
    return (df["latitude"].between(BOX["min_lat"], BOX["max_lat"])
            & df["longitude"].between(BOX["min_lon"], BOX["max_lon"]))


# ---------------- main ----------------
def main():
    # ---- donor: global minus Himalaya, split at the SAME timestamps ----
    g = pd.read_parquet("comcat_global_m4_features.parquet")
    g = g.sort_values("time").reset_index(drop=True)
    spl = np.load(MODELS_DIR / "comcat_global_m4__splits.npz")
    b1, b2, b3 = int(spl["b1"]), int(spl["b2"]), int(spl["b3"])
    t_train_end = g["time"].iloc[b1]
    t_val_end = g["time"].iloc[b2]
    t_test_start = g["time"].iloc[b3]

    him_mask = in_box(g).to_numpy()
    donor = g[~him_mask].reset_index(drop=True)
    td = donor["time"]
    d1 = int((td < t_train_end).sum())
    d2 = int((td < t_val_end).sum())
    d3 = int((td < t_test_start).sum())
    print(f"donor events: {len(donor)} (removed {him_mask.sum()} "
          f"Himalayan events)")

    feats = [f for f in FEATURES if f in donor.columns]
    yD = donor[REF_LABEL].to_numpy(np.int8)

    from sklearn.experimental import enable_iterative_imputer  # noqa
    from sklearn.impute import IterativeImputer
    imp = IterativeImputer(max_iter=10, random_state=SEED)
    imp.fit(donor[feats].iloc[:d1])
    XD = pd.DataFrame(imp.transform(donor[feats]), columns=feats)\
           .astype(np.float32)

    best = json.loads((MODELS_DIR / "comcat_global_m4__optuna_best.json")
                      .read_text())["xgboost"]
    from xgboost import XGBClassifier
    ck = MODELS_DIR / "donor_minus_himalaya__xgboost.json"
    donor_model = XGBClassifier(**best, random_state=SEED, n_jobs=-1,
                                tree_method="hist",
                                device="cuda" if GPU else "cpu",
                                eval_metric="logloss")
    if ck.exists():
        donor_model.load_model(ck)
        print("donor model loaded from checkpoint")
    else:
        donor_model.fit(XD.iloc[:d1], yD[:d1])
        donor_model.save_model(ck)
        print("donor model trained + checkpointed")

    pD = donor_model.predict_proba(XD)[:, 1].astype(np.float64)
    isoD = isotonic_fit(pD[d1:d2], yD[d1:d2])
    pD_cal = isoD.predict(pD[d2:d3])
    yD_cal = yD[d2:d3]

    # ---- target: Himalaya events in the donor test period ----
    h = pd.read_parquet("comcat_himalaya_m4_features.parquet")
    h = h.sort_values("time").reset_index(drop=True)
    hspl = np.load(MODELS_DIR / "comcat_himalaya_m4__splits.npz")
    h2, h3 = int(hspl["b2"]), int(hspl["b3"])
    yH = h[REF_LABEL].to_numpy(np.int8)

    target_mask = (h["time"] >= t_test_start).to_numpy()
    tgt_idx = np.where(target_mask)[0]
    yT = yH[tgt_idx]
    print(f"target events (Himalaya, donor-test period): {len(tgt_idx)} "
          f"(mainshock rate {yT.mean():.3f})")
    if len(tgt_idx) < 100:
        print("[warn] very small target set — report with caution")

    XT = pd.DataFrame(imp.transform(h[feats].iloc[tgt_idx]),
                      columns=feats).astype(np.float32)
    pT_raw = donor_model.predict_proba(XT)[:, 1].astype(np.float64)

    rows = []

    # ---- 1) native ----
    pN = np.load(MODELS_DIR /
                 "comcat_himalaya_m4__xgboost__recal_probs.npz")["prob"]\
           .astype(np.float64)
    cal_idx = np.arange(h2, h3)
    cal_idx = cal_idx[~np.isin(cal_idx, tgt_idx)]
    rows.append({"system": "native (Himalaya-trained)",
                 "n_target": len(tgt_idx),
                 **discrimination(yT, pN[tgt_idx]),
                 **mondrian_eval(pN[cal_idx], yH[cal_idx],
                                 pN[tgt_idx], yT)})

    # ---- 2) zero-adaptation transfer ----
    pT_donorrecal = isoD.predict(pT_raw)
    rows.append({"system": "transfer (donor calib only)",
                 "n_target": len(tgt_idx),
                 **discrimination(yT, pT_donorrecal),
                 **mondrian_eval(pD_cal, yD_cal, pT_donorrecal, yT)})

    # ---- 3) transfer + target adaptation ----
    ta_mask = ((h["time"] < t_test_start).to_numpy()
               & (np.arange(len(h)) >= h2))
    ta_idx = np.where(ta_mask)[0]
    XC = pd.DataFrame(imp.transform(h[feats].iloc[ta_idx]),
                      columns=feats).astype(np.float32)
    pC_raw = donor_model.predict_proba(XC)[:, 1].astype(np.float64)
    yC = yH[ta_idx]
    isoT = isotonic_fit(pC_raw, yC)
    rows.append({"system": f"transfer + target adaptation "
                           f"(n_cal={len(ta_idx)})",
                 "n_target": len(tgt_idx),
                 **discrimination(yT, isoT.predict(pT_raw)),
                 **mondrian_eval(isoT.predict(pC_raw), yC,
                                 isoT.predict(pT_raw), yT)})

    out = pd.DataFrame(rows).round(4)
    out.to_csv("step10_transfer.csv", index=False)
    print(f"\n=== step10_transfer.csv (target: Himalaya, alpha={ALPHA}) ===")
    print(out.to_string(index=False))


if __name__ == "__main__":
    main()

GPU available: True
donor events: 537129 (removed 4688 Himalayan events)


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


donor model trained + checkpointed


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [16:43:42] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


target events (Himalaya, donor-test period): 732 (mainshock rate 0.601)

=== step10_transfer.csv (target: Himalaya, alpha=0.1) ===
                                  system  n_target    auc     f1    ece  brier  cov_class1  cov_class0  avg_set_size  abstain_rate
               native (Himalaya-trained)       732 0.9386 0.9365 0.0214 0.0664      0.8705      0.9486        1.2117        0.2117
             transfer (donor calib only)       732 0.9339 0.9412 0.0226 0.0654      0.9773      0.8459        1.0260        0.0260
transfer + target adaptation (n_cal=675)       732 0.9317 0.9386 0.0386 0.0684      0.9000      0.9041        1.1366        0.1366


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

REF_LABEL = "label_R50_T30"
FULL = ["mw", "depth", "gap", "rms", "dmin", "magError",
        "local_count_7d", "log_energy_30d", "mag_diff_30d",
        "time_gap_local", "seismicity_z", "latitude", "longitude"]
ABLATIONS = {
    "full": FULL,
    "no_magdiff": [f for f in FULL if f != "mag_diff_30d"],
    "magdiff_only": ["mag_diff_30d"],
    "no_mw_family": [f for f in FULL
                     if f not in ("mag_diff_30d", "mw", "log_energy_30d")],
}
CATS = ["comcat_global_m4", "scedc_socal_m25", "comcat_himalaya_m4"]
MODELS_DIR = Path("./models")
N_BOOT = 2000
SEED = 42

try:
    import subprocess
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    GPU = False
print(f"GPU available: {GPU}")


# ----------------------------------------------------------------------------
# DeLong test for two correlated AUCs (fast implementation)
# ----------------------------------------------------------------------------
def _midrank(x):
    order = np.argsort(x)
    z = x[order]
    n = len(x)
    t = np.zeros(n)
    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n)
    out[order] = t
    return out


def delong_test(y, p1, p2):
    """Two-sided p-value for H0: AUC(p1) == AUC(p2) on the same data
    (DeLong, DeLong & Clarke-Pearson 1988). Returns (auc1, auc2, p)."""
    from scipy.stats import norm
    pos, neg = p1[y == 1], p1[y == 0]
    m, n = len(pos), len(neg)
    preds = np.vstack([p1, p2])
    k = 2
    tx = np.empty((k, m))
    ty = np.empty((k, n))
    tz = np.empty((k, m + n))
    for r in range(k):
        px, py = preds[r][y == 1], preds[r][y == 0]
        tz[r] = _midrank(np.concatenate([px, py]))
        tx[r] = _midrank(px)
        ty[r] = _midrank(py)
    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    S = sx / m + sy / n
    d = np.array([1.0, -1.0])
    var = d @ S @ d
    if var <= 0:
        return float(aucs[0]), float(aucs[1]), 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    p = 2 * norm.sf(abs(z))
    return float(aucs[0]), float(aucs[1]), float(p)


# ----------------------------------------------------------------------------
# Bootstrap CIs
# ----------------------------------------------------------------------------
def ece(y, p, n_bins=15):
    order = np.argsort(p)
    y, p = y[order], p[order]
    e = 0.0
    for b in np.array_split(np.arange(len(p)), n_bins):
        if len(b):
            e += (len(b) / len(p)) * abs(y[b].mean() - p[b].mean())
    return float(e)


def boot_ci(stat_fn, y, *arrays, n_boot=N_BOOT, seed=SEED):
    """Stratified bootstrap 95% CI for stat_fn(y, *arrays)."""
    rng = np.random.default_rng(seed)
    idx1, idx0 = np.where(y == 1)[0], np.where(y == 0)[0]
    vals = np.empty(n_boot)
    for b in range(n_boot):
        s = np.concatenate([rng.choice(idx1, len(idx1), replace=True),
                            rng.choice(idx0, len(idx0), replace=True)])
        vals[b] = stat_fn(y[s], *[a[s] for a in arrays])
    return (float(stat_fn(y, *arrays)),
            float(np.percentile(vals, 2.5)),
            float(np.percentile(vals, 97.5)))


def cov_class(y, covered, cls):
    m = y == cls
    return float(covered[m].mean()) if m.any() else np.nan


# ----------------------------------------------------------------------------
# A) Ablation
# ----------------------------------------------------------------------------
def run_ablation():
    import joblib
    from xgboost import XGBClassifier
    from sklearn.metrics import roc_auc_score, f1_score, recall_score

    rows, probs_store = [], {}
    for cat in CATS:
        spl = np.load(MODELS_DIR / f"{cat}__splits.npz")
        b1, b3 = int(spl["b1"]), int(spl["b3"])
        df = pd.read_parquet(f"{cat}_features.parquet")
        df = df.sort_values("time").reset_index(drop=True)
        y = df[REF_LABEL].to_numpy(np.int8)
        yte = y[b3:]
        imp = joblib.load(MODELS_DIR / f"{cat}__imputer.joblib")
        feats_all = [f for f in FULL if f in df.columns]
        Xi = pd.DataFrame(imp.transform(df[feats_all].astype(np.float32)),
                          columns=feats_all).astype(np.float32)
        best = json.loads((MODELS_DIR / f"{cat}__optuna_best.json")
                          .read_text())["xgboost"]

        for abl, feats in ABLATIONS.items():
            feats = [f for f in feats if f in Xi.columns]
            ck = MODELS_DIR / f"{cat}__abl_{abl}__probs.npz"
            if ck.exists():
                prob = np.load(ck)["prob"].astype(np.float64)
            else:
                m = XGBClassifier(**best, random_state=SEED, n_jobs=-1,
                                  tree_method="hist",
                                  device="cuda" if GPU else "cpu",
                                  eval_metric="logloss")
                m.fit(Xi[feats].iloc[:b1], y[:b1])
                prob = m.predict_proba(Xi[feats])[:, 1].astype(np.float64)
                np.savez(ck, prob=prob.astype(np.float32))
            probs_store[(cat, abl)] = (yte, prob[b3:])
            rows.append({"catalog": cat, "ablation": abl,
                         "n_features": len(feats),
                         "auc": roc_auc_score(yte, prob[b3:]),
                         "f1": f1_score(yte, (prob[b3:] >= 0.5).astype(int)),
                         "recall": recall_score(
                             yte, (prob[b3:] >= 0.5).astype(int))})
            print(f"  {cat} / {abl}: AUC {rows[-1]['auc']:.4f} "
                  f"F1 {rows[-1]['f1']:.4f}")

        # DeLong vs full
        yfull, pfull = probs_store[(cat, "full")]
        for abl in ABLATIONS:
            if abl == "full":
                continue
            _, pabl = probs_store[(cat, abl)]
            _, _, pval = delong_test(yfull, pfull, pabl)
            for r in rows:
                if r["catalog"] == cat and r["ablation"] == abl:
                    r["delong_p_vs_full"] = pval
    out = pd.DataFrame(rows).round(5)
    out.to_csv("step11_ablation.csv", index=False)
    print("\n=== step11_ablation.csv ===")
    print(out.to_string(index=False))
    return probs_store


# ----------------------------------------------------------------------------
# B) Significance battery
# ----------------------------------------------------------------------------
def run_significance(probs_store):
    from sklearn.metrics import roc_auc_score, f1_score
    rows = []

    for cat in CATS:
        spl = np.load(MODELS_DIR / f"{cat}__splits.npz")
        b2, b3 = int(spl["b2"]), int(spl["b3"])
        df = pd.read_parquet(f"{cat}_features.parquet",
                             columns=[REF_LABEL, "mag_diff_30d"])
        y = df[REF_LABEL].to_numpy(np.int8)
        yte = y[b3:]
        p = np.load(MODELS_DIR / f"{cat}__xgboost__recal_probs.npz")["prob"]\
              .astype(np.float64)
        pte, pcal = p[b3:], p[b2:b3]
        ycal = y[b2:b3]
        heur = (df["mag_diff_30d"].to_numpy() >= 0).astype(float)[b3:]

        # DeLong: model vs heuristic
        a1, a2, pv = delong_test(yte, pte, heur)
        rows.append({"catalog": cat, "test": "DeLong AUC: xgboost vs "
                     "heuristic", "stat_1": a1, "stat_2": a2, "p": pv})

        # bootstrap CIs: AUC, F1, ECE
        for name, fn, arr in [
                ("auc", lambda yy, pp: roc_auc_score(yy, pp), pte),
                ("f1", lambda yy, pp: f1_score(yy, (pp >= .5).astype(int)),
                 pte),
                ("ece", ece, pte)]:
            v, lo, hi = boot_ci(fn, yte, arr)
            rows.append({"catalog": cat, "test": f"bootstrap 95% CI: {name}",
                         "stat_1": v, "ci_lo": lo, "ci_hi": hi})

        # bootstrap CIs: Mondrian per-class coverage on the test block
        def q(scores, alpha=0.10):
            n = len(scores)
            qq = np.ceil((n + 1) * (1 - alpha)) / n
            return np.quantile(scores, min(qq, 1.0), method="higher")
        q1 = q(1 - pcal[ycal == 1])
        q0 = q(pcal[ycal == 0])
        covered = np.where(yte == 1, (1 - pte) <= q1, pte <= q0)\
                    .astype(float)
        for cls in (1, 0):
            v, lo, hi = boot_ci(lambda yy, cc: cov_class(yy, cc, cls),
                                yte, covered)
            rows.append({"catalog": cat,
                         "test": f"bootstrap 95% CI: coverage class {cls}",
                         "stat_1": v, "ci_lo": lo, "ci_hi": hi})

    # transfer experiment CIs + DeLong (native vs transfer), if step 10 ran
    tf = Path("step10_transfer.csv")
    if tf.exists() and (MODELS_DIR /
                        "donor_minus_himalaya__xgboost.json").exists():
        print("[note] transfer CIs use step-10 saved artifacts; native-vs-"
              "transfer DeLong requires the step-10 probability vectors — "
              "rerun step 10 with SAVE_PROBS=True if not present.")

    out = pd.DataFrame(rows).round(5)
    out.to_csv("step11_significance.csv", index=False)
    print("\n=== step11_significance.csv ===")
    print(out.to_string(index=False))


if __name__ == "__main__":
    store = run_ablation()
    run_significance(store)

GPU available: True
  comcat_global_m4 / full: AUC 0.9547 F1 0.8937
  comcat_global_m4 / no_magdiff: AUC 0.9521 F1 0.8855
  comcat_global_m4 / magdiff_only: AUC 0.9409 F1 0.8888
  comcat_global_m4 / no_mw_family: AUC 0.9013 F1 0.7954
  scedc_socal_m25 / full: AUC 0.9784 F1 0.9066
  scedc_socal_m25 / no_magdiff: AUC 0.9753 F1 0.8889
  scedc_socal_m25 / magdiff_only: AUC 0.9724 F1 0.8835
  scedc_socal_m25 / no_mw_family: AUC 0.9416 F1 0.8081
  comcat_himalaya_m4 / full: AUC 0.9378 F1 0.9393
  comcat_himalaya_m4 / no_magdiff: AUC 0.9361 F1 0.9234
  comcat_himalaya_m4 / magdiff_only: AUC 0.9388 F1 0.9393
  comcat_himalaya_m4 / no_mw_family: AUC 0.9158 F1 0.8845

=== step11_ablation.csv ===
           catalog     ablation  n_features     auc      f1  recall  delong_p_vs_full
  comcat_global_m4         full          13 0.95465 0.89370 0.97513               NaN
  comcat_global_m4   no_magdiff          12 0.95214 0.88548 0.96126           0.00000
  comcat_global_m4 magdiff_only           1 0.9

In [ ]:
!python step9_figures.py

saved fig1_completeness_splits
saved fig2_label_rate_heatmaps
saved fig3_flip_rate_heatmaps
saved fig4_reliability
saved fig5_aci_rolling
saved fig6_h1_sensitivity
saved fig7_trust_gradient
saved fig8_case_study_timeline
saved fig9_decision_costs
saved fig10_shap_stability
saved fig11_transfer

All figures in ./figures/ (.png + .pdf)
